# Live Demo — Fire and Smoke Detection

Runs both models (YOLOv8s and Faster R-CNN) on 6 representative D-Fire test images
and produces side-by-side visualizations (GT | YOLO | FRCN).

**Output**: `/kaggle/working/demo/01_fire_and_smoke.jpg` ... `06_cloud_scene.jpg`

In [ ]:
!pip install -q ultralytics==8.3.221

In [ ]:
import sys
import json
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image, ImageDraw, ImageFont

# Config
CKPT_DIR = "/kaggle/input/datasets/c3m3nt/cv-project-checkpoints"
TEST_IMAGES_DIR = "/kaggle/input/datasets/linxel/dfire-prepared/data/test/images"
TEST_LABELS_DIR = "/kaggle/input/datasets/linxel/dfire-prepared/data/test/labels"
OUT_DIR = "/kaggle/working/demo"

YOLO_THR = 0.30
FRCN_THR = 0.55
TARGET_H = 720   # resize height for display
RANDOM_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
ckpt_dir = Path(CKPT_DIR)
required = [
    ckpt_dir / "yolo_selected_best.pt",
    ckpt_dir / "faster_rcnn_best.pth",
    ckpt_dir / "final_yolo_protocol.py",
    ckpt_dir / "budget_protocol.py",
]
for p in required:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {p}")
    assert p.exists(), f"Not found: {p}"

assert Path(TEST_IMAGES_DIR).is_dir(), f"Not found: {TEST_IMAGES_DIR}"
assert Path(TEST_LABELS_DIR).is_dir(), f"Not found: {TEST_LABELS_DIR}"

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"\nOut dir: {OUT_DIR}")

# add protocol dir to sys.path so `final_yolo_protocol` is importable
if str(ckpt_dir) not in sys.path:
    sys.path.insert(0, str(ckpt_dir))
print("sys.path updated.")

In [ ]:
CLASS_NAMES = {1: "smoke", 2: "fire"}
COLORS = {1: (0, 200, 255), 2: (255, 80, 80)}   # smoke=cyan, fire=red


import matplotlib
from pathlib import Path

CLASS_NAMES = {1: "smoke", 2: "fire"}
COLORS = {1: (0, 200, 255), 2: (255, 80, 80)}

_FONT_PATHS = [
    str(Path(matplotlib.get_data_path()) / "fonts" / "ttf" / "DejaVuSans-Bold.ttf"),
    str(Path(matplotlib.get_data_path()) / "fonts" / "ttf" / "DejaVuSans.ttf"),
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
]
_FONT_PATH = next((p for p in _FONT_PATHS if Path(p).exists()), None)

if _FONT_PATH is None:
    print("WARNING: no TTF font found — text will be small (PIL default)")
else:
    print(f"Using font: {_FONT_PATH}")


def _try_font(size):
    """TrueType font at exact size; verified size attribute."""
    if _FONT_PATH is not None:
        try:
            font = ImageFont.truetype(_FONT_PATH, size)
            assert getattr(font, "size", None) == size, (
                f"font size mismatch: got {getattr(font, 'size', None)}, want {size}"
            )
            return font
        except (OSError, AssertionError):
            pass
    return ImageFont.load_default()


# Sanity check
_test = _try_font(200)
print(f"Test font: {type(_test).__name__}, size={getattr(_test, 'size', 'n/a')}")


def load_gt(label_path, img_size):
    """Load YOLO-format label -> (boxes_xyxy [N,4], labels [N] in 1=smoke, 2=fire)."""
    w, h = img_size
    boxes, labels = [], []
    label_path = Path(label_path)
    if not label_path.exists():
        return torch.zeros((0, 4)), torch.zeros((0,), dtype=torch.long)

    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cls, xc, yc, bw, bh = map(float, parts)
        x1 = max(0.0, (xc - bw / 2) * w)
        y1 = max(0.0, (yc - bh / 2) * h)
        x2 = min(w, (xc + bw / 2) * w)
        y2 = min(h, (yc + bh / 2) * h)
        if x2 <= x1 or y2 <= y1:
            continue
        boxes.append([x1, y1, x2, y2])
        labels.append(1 if int(cls) == 0 else 2)

    if not boxes:
        return torch.zeros((0, 4)), torch.zeros((0,), dtype=torch.long)
    return (torch.tensor(boxes, dtype=torch.float32),
            torch.tensor(labels, dtype=torch.long))


def draw_boxes_on(img, boxes, scores, labels, thr, font, thickness=6):
    """Draw all boxes with score >= thr on PIL image (in place)."""
    draw = ImageDraw.Draw(img)
    # derive text offset from font size so the label doesn't overlap
    try:
        font_size = font.size
    except AttributeError:
        font_size = 26
    for b, s, lab in zip(boxes, scores, labels):
        if float(s) < thr:
            continue
        x1, y1, x2, y2 = map(int, b)
        color = COLORS.get(int(lab), (255, 255, 0))
        draw.rectangle([x1, y1, x2, y2], outline=color, width=thickness)
        name = CLASS_NAMES.get(int(lab), str(int(lab)))
        text = f"{name} {float(s):.2f}"
        bbox = draw.textbbox((x1, max(0, y1 - font_size - 6)), text, font=font)
        draw.rectangle(bbox, fill=color)
        draw.text((x1, max(0, y1 - font_size - 6)), text,
                  fill=(0, 0, 0), font=font)
    return img

In [ ]:
def make_side_by_side(img_path, yolo_out, frcn_out,
                      yolo_thr, frcn_thr,
                      gt_boxes=None, gt_labels=None,
                      out_path=None, title=None):
    """
    Layout:
        [        Ground truth (centered)        ]
        [ YOLOv8s @ thr ] [ Faster R-CNN @ thr ]
    Top row: GT (larger). Bottom row: two models side by side.
    """
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    scale = TARGET_H / h
    new_w = int(w * scale)
    panel_resized = img.resize((new_w, TARGET_H), Image.BILINEAR)

    title_size = int(TARGET_H * 0.055)
    box_size   = int(TARGET_H * 0.05)
    panel_font = _try_font(title_size)   # "Ground truth", "YOLOv8s @ 0.30", etc.
    box_font   = _try_font(box_size)     # "smoke 0.79"

    row_gap   = 30             # vertical gap between top and bottom rows
    col_gap   = 20             # horizontal gap between the 2 bottom panels
    margin    = 20
    title_h   = 120             # space for the main title at the top
    panel_label_h = 44         # space for the label above each panel

    bottom_w = 2 * new_w + col_gap
    gt_panel_w = new_w
    gt_panel_h = TARGET_H

    total_w = bottom_w + 2 * margin
    total_h = title_h + panel_label_h + gt_panel_h + row_gap + panel_label_h + TARGET_H + margin

    canvas = Image.new("RGB", (total_w, total_h), "white")
    draw = ImageDraw.Draw(canvas)

    
    if title:
        draw.text((margin, 22), title, fill="black", font=panel_font)

    gt_img = img.resize((gt_panel_w, gt_panel_h), Image.BILINEAR)
    gt_scale_x = gt_panel_w / w
    gt_scale_y = gt_panel_h / h

    if gt_boxes is not None and len(gt_boxes) > 0:
        dg = ImageDraw.Draw(gt_img)
        for b, lab in zip(gt_boxes, gt_labels):
            x1 = int(b[0] * gt_scale_x); y1 = int(b[1] * gt_scale_y)
            x2 = int(b[2] * gt_scale_x); y2 = int(b[3] * gt_scale_y)
            color = (0, 200, 0)
            dg.rectangle([x1, y1, x2, y2], outline=color, width=6)
            name = CLASS_NAMES.get(int(lab), str(int(lab)))
            text = f"GT {name}"
            bbox = dg.textbbox((x1, max(0, y1 - box_size - 6)), text, font=box_font)
            dg.rectangle(bbox, fill=color)
            dg.text((x1, max(0, y1 - box_size - 6)), text, fill=(0, 0, 0), font=box_font)

    gt_x = margin + (bottom_w - gt_panel_w) // 2   # centered (same width → 0 offset)
    gt_y = title_h
    canvas.paste(gt_img, (gt_x, gt_y))
    # Label above GT panel
    draw.text((gt_x, gt_y - 6), "Ground truth",
              fill=(0, 130, 0), font=panel_font, anchor="lb")

    bottom_y = gt_y + gt_panel_h + row_gap + panel_label_h

    # YOLO panel
    yolo_x = margin
    yolo_img = panel_resized.copy()
    draw_boxes_on(yolo_img,
                  yolo_out["boxes"].numpy() * scale,
                  yolo_out["scores"].numpy(),
                  yolo_out["labels"].numpy(),
                  thr=yolo_thr, font=box_font)
    canvas.paste(yolo_img, (yolo_x, bottom_y))
    draw.text((yolo_x, bottom_y - 6), f"YOLOv8s @ {yolo_thr:.2f}",
              fill=(150, 0, 150), font=panel_font, anchor="lb")

    # FRCN panel
    frcn_x = margin + new_w + col_gap
    frcn_img = panel_resized.copy()
    draw_boxes_on(frcn_img,
                  frcn_out["boxes"].numpy() * scale,
                  frcn_out["scores"].numpy(),
                  frcn_out["labels"].numpy(),
                  thr=frcn_thr, font=box_font)
    canvas.paste(frcn_img, (frcn_x, bottom_y))
    draw.text((frcn_x, bottom_y - 6), f"Faster R-CNN @ {frcn_thr:.2f}",
              fill=(0, 100, 200), font=panel_font, anchor="lb")

    if out_path:
        canvas.save(out_path, quality=92)
        print(f"  saved -> {out_path}")
    return canvas

In [ ]:
def select_demo_images(images_dir, labels_dir, seed=RANDOM_SEED):
    """Pick 6 representative images across categories."""
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)
    imgs = sorted(p for p in images_dir.iterdir()
                  if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

    buckets = defaultdict(list)
    empty_pool = []

    for p in imgs:
        lbl = labels_dir / (p.stem + ".txt")
        classes = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if parts:
                    classes.append(int(parts[0]))
        n_smoke = classes.count(0)
        n_fire = classes.count(1)

        if n_smoke > 0 and n_fire > 0:
            buckets["fire_and_smoke"].append(p)
        elif n_smoke > 0:
            buckets["smoke_only"].append(p)
        elif n_fire > 0:
            buckets["fire_only"].append(p)
        elif len(classes) == 0:
            empty_pool.append(p)

    print(f"[pool] fire_and_smoke: {len(buckets['fire_and_smoke'])}")
    print(f"[pool] smoke_only:     {len(buckets['smoke_only'])}")
    print(f"[pool] fire_only:      {len(buckets['fire_only'])}")
    print(f"[pool] empty:          {len(empty_pool)}")

    rng = np.random.default_rng(seed)
    selected = []

    def pick(cat, n=1):
        pool = buckets[cat]
        if not pool:
            print(f"  WARNING: empty bucket {cat}")
            return
        for i in rng.choice(len(pool), size=min(n, len(pool)), replace=False):
            selected.append((cat, pool[i]))

    pick("fire_and_smoke", 1)
    pick("smoke_only", 1)
    pick("fire_only", 1)

    # Small fire: any fire_only image with fire box area < 2% of image
    small = []
    for p in buckets["fire_only"]:
        try:
            with Image.open(p) as im:
                W, H = im.size
        except Exception:
            continue
        for line in (labels_dir / (p.stem + ".txt")).read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            if float(parts[3]) * float(parts[4]) < 0.02:
                small.append(p)
                break
    if small:
        selected.append(("small_fire", small[rng.integers(len(small))]))
        print(f"[pick] small_fire:     {len(small)}")
    else:
        print("[pick] small_fire:     fallback to fire_only")
        pick("fire_only", 1)

    # Bright warm scene (potential fire FP) — high R, low B
    bright = []
    for path in empty_pool:
        try:
            with Image.open(path) as im:
                arr = np.asarray(im.convert("RGB").resize((64, 64)))
        except Exception:
            continue
        m = arr.reshape(-1, 3).mean(axis=0)
        if m[0] > 150 and m[0] > m[2] + 30:
            bright.append(path)
    if bright:
        selected.append(("bright_scene", bright[rng.integers(len(bright))]))
        print(f"[pick] bright_scene:   {len(bright)}")
    else:
        print("[pick] bright_scene:   no warm-toned empties found")
        if empty_pool:
            selected.append(("empty", empty_pool[rng.integers(len(empty_pool))]))

    # Cloud scene — cool/grey empty
    used = {p for _, p in selected}
    cloud = []
    for path in empty_pool:
        if path in used:
            continue
        try:
            with Image.open(path) as im:
                arr = np.asarray(im.convert("RGB").resize((64, 64)))
        except Exception:
            continue
        m = arr.reshape(-1, 3).mean(axis=0)
        if m[2] > m[0] - 10 and m.mean() > 100:
            cloud.append(path)
    if cloud:
        selected.append(("cloud_scene", cloud[rng.integers(len(cloud))]))
        print(f"[pick] cloud_scene:    {len(cloud)}")

    # Fill to 6 with any remaining empties
    while len(selected) < 6 and empty_pool:
        used = {p for _, p in selected}
        rest = [p for p in empty_pool if p not in used]
        if not rest:
            break
        selected.append(("empty", rest[rng.integers(len(rest))]))
        print(f"[pick] empty fallback: {selected[-1][1].name}")

    return selected[:6]

In [ ]:
# YOLO
from ultralytics import YOLO
yolo_ckpt = ckpt_dir / "yolo_selected_best.pt"
yolo_wrapper = YOLO(str(yolo_ckpt))
yolo_core = yolo_wrapper.model.to(DEVICE).float().eval()
print(f"[YOLO] classes={yolo_wrapper.names}, nc={yolo_core.nc}")
assert yolo_core.nc == 2

# FRCN
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

frcn_ckpt = ckpt_dir / "faster_rcnn_best.pth"
frcn_model = fasterrcnn_resnet50_fpn(weights=None)
in_f = frcn_model.roi_heads.box_predictor.cls_score.in_features
frcn_model.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes=3)
frcn_model.transform.min_size = (640,)
frcn_model.transform.max_size = 640
ckpt = torch.load(str(frcn_ckpt), map_location="cpu")
frcn_model.load_state_dict(ckpt["model_state_dict"])
frcn_model.to(DEVICE).eval()
print(f"[FRCN] epoch={ckpt.get('epoch', '?')}")

print("Models loaded.")

In [ ]:
from final_yolo_protocol import predict_with_baseline_input


@torch.no_grad()
def predict_yolo(img):
    """Returns boxes (xyxy in original pixels), scores, labels (1=smoke, 2=fire)."""
    out = predict_with_baseline_input(
        yolo_core, [img], device=DEVICE,
        score_floor=0.001, nms_iou=0.7, max_det=100,
    )[0]
    return {
        "boxes": out["boxes"].cpu(),
        "scores": out["scores"].cpu(),
        "labels": (out["labels"].cpu() + 1).long(),
    }


@torch.no_grad()
def predict_frcn(img):
    t = T.ToTensor()(img).to(DEVICE)
    out = frcn_model([t])[0]
    return {
        "boxes": out["boxes"].cpu(),
        "scores": out["scores"].cpu(),
        "labels": out["labels"].cpu(),   # bg=0, smoke=1, fire=2 — same convention
    }

In [ ]:
# Select 6 demo images
print("Selecting 6 representative images")
selection = select_demo_images(TEST_IMAGES_DIR, TEST_LABELS_DIR)
print(f"\nSelected {len(selection)} images:")
for i, (cat, p) in enumerate(selection, 1):
    print(f"  [{i}] {cat:15s}  {p.name}")

# Run inference + save side-by-side
print("Running inference")
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

results = []
t_start = time.time()

for i, (cat, path) in enumerate(selection, start=1):
    print(f"\n[{i}/6] {cat}: {path.name}")
    img = Image.open(path).convert("RGB")
    W, H = img.size
    gt_boxes, gt_labels = load_gt(
        Path(TEST_LABELS_DIR) / (path.stem + ".txt"), (W, H)
    )

    t0 = time.time()
    yolo_out = predict_yolo(img)
    t_yolo = time.time() - t0

    t0 = time.time()
    frcn_out = predict_frcn(img)
    t_frcn = time.time() - t0

    n_yolo = int((yolo_out["scores"] >= YOLO_THR).sum())
    n_frcn = int((frcn_out["scores"] >= FRCN_THR).sum())
    print(f"    YOLO: {n_yolo} boxes ({t_yolo*1000:.0f} ms)  |  "
          f"FRCN: {n_frcn} boxes ({t_frcn*1000:.0f} ms)")

    out_file = out_dir / f"{i:02d}_{cat}.jpg"
    make_side_by_side(
        img_path=path,
        yolo_out=yolo_out,
        frcn_out=frcn_out,
        yolo_thr=YOLO_THR,
        frcn_thr=FRCN_THR,
        gt_boxes=gt_boxes.numpy() if len(gt_boxes) else None,
        gt_labels=gt_labels.numpy() if len(gt_labels) else None,
        out_path=str(out_file),
        title=f"[{i}/6] {cat.replace('_', ' ').title()} — {path.name}",
    )

    results.append({
        "index": i,
        "category": cat,
        "image": str(path),
        "yolo_boxes_at_thr": n_yolo,
        "frcn_boxes_at_thr": n_frcn,
        "gt_boxes": int(len(gt_boxes)),
        "yolo_ms": round(t_yolo * 1000, 1),
        "frcn_ms": round(t_frcn * 1000, 1),
        "out_file": str(out_file),
    })

elapsed = time.time() - t_start

# Save summary
summary = {
    "n_images": len(selection),
    "yolo_threshold": YOLO_THR,
    "frcn_threshold": FRCN_THR,
    "total_seconds": round(elapsed, 2),
    "per_image_seconds": round(elapsed / max(len(selection), 1), 2),
    "results": results,
}
(out_dir / "demo_summary.json").write_text(json.dumps(summary, indent=2))

print(f"DONE ({elapsed:.1f} s total)")
for r in results:
    print(f"  {Path(r['out_file']).name:30s}  "
          f"GT={r['gt_boxes']:2d}  YOLO={r['yolo_boxes_at_thr']:2d}  FRCN={r['frcn_boxes_at_thr']:2d}")

In [ ]:
import shutil

zip_path = "/kaggle/working/demo_outputs"
shutil.make_archive(zip_path, "zip", OUT_DIR)
print(f"ZIP: {zip_path}.zip")

# list contents
for f in sorted(Path(OUT_DIR).iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:40s} {size_kb:7.1f} KB")